This notebook generates data located in `mcmc_data/` for the **ball-drop** example discussed in *"Bayesian model–data comparison incorporating theoretical uncertainties"* ([arXiv:2504.13144](https://arxiv.org/abs/2504.13144)).  

Computed data already exists in `mcmc_data/`. One can skip running this file and run `plots.ipynb`notebook to generate plots using the **precomputed MCMC chains**, or run this file to regenerate the data. This may take some time.


# Ball drop experiment

A ball is dropped from a tower of height $h_0$. The height from ground, velocity and acceleration of the ball is recorded at discrete time points until it hits the ground.

The aim is to extract the value of accelaration due to gravity, $g$.

Reality has air resistance. Physics theory ignores air resistance.

In [ ]:
import os, sys
import numpy as np
from scipy.stats import beta
from scipy import optimize
from sklearn.gaussian_process.kernels import DotProduct, RBF

from models import BallDrop
from quantiles import quantiles_model, quantiles_modelPlusGP

# Add src directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "../../..", "src"))
sys.path.append(src_dir)
from ModelDiscrepancy import ModelDiscrepancy as MD
from sampling_methods import pocomc_sampling as pocomc
from plot_scripts import setup_rc_params
setup_rc_params()

import matplotlib.pyplot as plt
# import corner
# import seaborn as sns; sns.set_style("darkgrid")#; sns.set_context("talk")

In [ ]:
import logging
from pathlib import Path

def setup_logging(logfile="run.log", level=logging.INFO):
    """Configure root logger to log to both console and file."""
    log_path = Path(logfile)
    log_path.parent.mkdir(parents=True, exist_ok=True)   # <-- make parent dirs

    # Clear any existing handlers
    logging.getLogger().handlers.clear()

    # Configure logging
    logging.basicConfig(
        level=level,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.StreamHandler(),                  # console
            logging.FileHandler(log_path, mode="w")   # file
        ]
    )


## Define True theory and generate "experimental" observations

In [ ]:
m_true, D_true = 1.0, 0.1  # mass and diameter of ball
beta_true, gamma_true = 0.0, 40.0  # coefficient of linear and quadratic drag

TrueModel = BallDrop(m=m_true, D=D_true, beta=beta_true, gamma=gamma_true)

# set parameters ------------------->
t0 = 0.0
h0_init = 60.0
v0_init = 0.0
g_true = 9.8

sig_h = 0.1
sig_v = 0.2
sig_a = 0.3

t_start = 0.0  # location of first data points 

np.random.seed(42)  # Set a random seed for reproducibility

# Generate 10 evenly spaced points between t_start and 1.0
t_obs = np.linspace(t_start, 1.0, 10)
# Add a random perturbations to t_obs
perturb = np.random.uniform(1e-3, 1e-2, size=t_obs.shape)
perturb[0] = 0.0  # No perturbation for the first element
t_obs = np.round(t_obs+perturb, decimals=4)

truth = TrueModel.height_velocity_accln(t=t_obs, g=g_true, v0=v0_init, h0=h0_init, t0=t0)

height = np.random.normal(truth[:, 0], sig_h)    # observed height at input times
velocity = np.random.normal(truth[:, 1], sig_v)  # observed velocity at input times
accln = np.random.normal(truth[:, 2], sig_a)     # observed acceleration at input times

err_h = np.array([sig_h]*len(t_obs))
err_v = np.array([sig_v]*len(t_obs))
err_a = np.array([sig_a]*len(t_obs))

# Save the data to experimental_data.dat
data = np.column_stack((t_obs, height, err_h, velocity, err_v, accln, err_a))
# Define a header string
header_str = "# t       height    err_h    velocity    err_v    accln    err_a"
np.savetxt("experimental_data.dat", data, fmt="%.6f", delimiter=" ", header=header_str, comments="")


In [ ]:
font_size = 11
# fig, axs = plt.subplots(1, 3, constrained_layout=False, figsize=(10, 2))
# plt.subplots_adjust(left=0.0, right=0.98, wspace=0.3)

fig, axs = plt.subplots(1, 3, figsize=(10, 2.2))
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.0, hspace=0.0)

# axs[0].plot(t_obs, height, color='r', marker="o")#, label=f'Truth')
axs[0].errorbar(t_obs, height, yerr=[1.96*err_h, 1.96*err_h], 
             fmt='o', markersize=3, capsize=3, color='black', label=f'``Experiment": 95\% CI')
axs[0].set_ylabel('height', fontsize = font_size)
axs[0].set_xlabel('time', fontsize = font_size)
axs[0].legend(loc='lower left', frameon=False, bbox_to_anchor=(-0.05, 0), markerscale=0.7, handletextpad=-0.2, labelspacing=0.0, fontsize=11)

axs[1].errorbar(t_obs, velocity, yerr=[1.96*err_v, 1.96*err_v], 
             fmt='o', markersize=3, capsize=3, color='black')
axs[1].set_ylabel('velocity', fontsize = font_size)
axs[1].set_xlabel('time', fontsize = font_size)

axs[2].errorbar(t_obs, accln, yerr=[1.96*err_a, 1.96*err_a], 
             fmt='o', markersize=3, capsize=3, color='black')
axs[2].set_ylabel('acceleration', fontsize = font_size)
axs[2].set_xlabel('time', fontsize = font_size)

# plt.savefig("BD_obs_exp.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Save model predictions from true theory for plotting later. 
Also defined is `t_plot` for making model predictions at those points

In [ ]:
t_plot = np.linspace(t_start, 1.3, 50)  # for getting quantiles for observable predictions
true_val = TrueModel.height_velocity_accln(t=t_plot, g=g_true, v0=v0_init, h0=h0_init, t0=t0)
data = np.column_stack((t_plot, true_val))

header_str = "# t        height      velocity     accln"
np.savetxt('truth.txt', data, header=header_str, fmt="%.8f")

## Define physics model

In [ ]:
class phy_model():
    def __init__(self, Model, observables, t_obs=t_obs, h0=60.0, t0=0.0):
        self.Model = Model
        self.observables = observables
        self.t_obs = t_obs
        self.h0 = h0
        self.t0 = t0

    def predict(self, theta):
        theta=np.atleast_1d(theta)
        g, v0 = theta[0], theta[1]
        sol = self.Model.height_velocity_accln(t=self.t_obs, g=g, v0=v0, h0=self.h0, t0=self.t0)
        
        # The order of observables as returned by the model:
        model_keys = list(observables.keys())
        active_keys = [key for key in model_keys if self.observables.get(key, False)]
    
        # Build dictionaries for predictions and errors.
        sol_dict = {key: sol[:, idx] for idx, key in enumerate(model_keys)}
        
        selected_predictions = [sol_dict[key] for key in active_keys]
        
        res = np.concatenate(selected_predictions)
        err_selected = np.zeros_like(res)
        
        return res, err_selected

    def predict_all(self, theta):
        theta=np.atleast_1d(theta)
        g, v0 = theta[0], theta[1]
        sol = self.Model.height_velocity_accln(t=self.t_obs, g=g, v0=v0, h0=self.h0, t0=self.t0)
        sol1 = np.concatenate(sol.T)
        err = np.zeros_like(sol1)
        
        return sol1, err


## Define discrepancy kernel and priors

In [ ]:
def MD_kernel(X1, X2, cbar, l, r, s):
    """
    Compute the covariance matrix using a custom kernel:
    K(X1, X2) = s^2 + cbar^2 * (X1 @ X2.T)^r * exp(-||X1 - X2||^2 / (2 * l^2)).

    This kernel serves as the core of the Model Discrepancy framework and should be modified
    to suit specific problems.

    Parameters:
        - X1 (numpy.ndarray): A matrix where each row represents an input vector.
        - X2 (numpy.ndarray): A matrix where each row represents an input vector.
        - cbar (float): The marginal variance parameter.
        - l (float): The length scale parameter for the RBF term.
        - r (float): The power applied to the dot product term.
        - s (float): A constant shift.

    Returns:
        - numpy.ndarray: A covariance matrix computed using the defined kernel.
    """
    # Compute the dot product term
    dot_product_term = np.dot(X1, X2.T) ** r if r > 0 else 1.0

    # Compute the RBF kernel term
    rbf_kernel = RBF(length_scale=l)
    rbf_term = rbf_kernel(X1, X2)

    # Combine terms
    cov_matrix =  s**2 + cbar**2 * dot_product_term * rbf_term
    return cov_matrix
    
# ===========================================================================================

# Define function for general beta priors 
def log_prior_param(param, alpha_val, beta_val, param_min, param_max):
    """
    Log prior for parameter based on a Beta distribution.
    """
    scale = param_max - param_min  # Rescaling factor

    if param_min < param < param_max:
        # Rescale param to the range [param_min, param_max]
        param_rescaled = (param - param_min) / scale
        
        prior = beta.pdf(param_rescaled, alpha_val, beta_val) / scale
        return np.log(prior)
    else:
        return -np.inf

# ===========================================================================================

model_param_bounds = np.array([[ 0.0,   20.0 ],
                               [-2.0 ,  2.0  ]])

# Define a list of log_prior functions for model parameters
log_prior_theta = [
    lambda param, i=i: log_prior_param(
        param, alpha_val=1.01, beta_val=1.01,  # for flat priors
        param_min=model_param_bounds[i][0], param_max=model_param_bounds[i][1]
    )
    for i in range(len(model_param_bounds))
]
# ===========================================================================================

HP_bounds = np.array([[ 0.0,   10.0 ],   # for cbar
                      [ 0.0 ,  10.0 ],   # for l
                      [ 0.0 ,  10.0 ],   # for r
                      [ 0.0 ,  10.0 ]])  # for s

# Define priors for GP hyperparmeters
log_prior_cbar = lambda param: log_prior_param(param, alpha_val=1.01, beta_val=1.01, param_min=HP_bounds[0][0], param_max=HP_bounds[0][1])
log_prior_l =    lambda param: log_prior_param(param, alpha_val=1.01, beta_val=1.01, param_min=HP_bounds[1][0], param_max=HP_bounds[1][1])
log_prior_r =    lambda param: log_prior_param(param, alpha_val=1.01, beta_val=1.01, param_min=HP_bounds[2][0], param_max=HP_bounds[2][1])
log_prior_s =    lambda param: log_prior_param(param, alpha_val=1.01, beta_val=1.01, param_min=HP_bounds[3][0], param_max=HP_bounds[3][1])


## Define function for MCMC sampling without and with model discrepancy (kernel I, II) given active observables

In [ ]:
# Settings for pocomc sampling -------->
n_effective_mltpl = 500
n_active_mltpl = 150  # 0.3 x n_eff
n_steps_mltpl = 2
save_every_n = 100

# ====================================================================================

def mcmc_woMD(exp_active_dict, model_predict, save_dir, n_total, resume):
    """
    Without model discrepancy
    """
    # Instantiate model discrepancy class
    md = MD(exp_data=exp_active_dict, 
            model_predict=model_predict, 
            log_priors_model=log_prior_theta,
            MD=False, MDkernels=None)
    
    # Model parameters bounds (for theta):
    min_param = model_param_bounds[:,0]
    max_param = model_param_bounds[:,1]
    lab = [r'$g$', r'$v_0$']
    
    num_param = len(min_param)
    n_effective = n_effective_mltpl * num_param
    n_active = n_active_mltpl * num_param
    n_steps = n_steps_mltpl * num_param
    
    samples = pocomc(min_param, max_param, 
                     log_posterior=md.log_posterior, samples_save_dir = f"{save_dir}", 
                     n_effective=n_effective, n_active=n_effective, 
                     n_steps=n_steps, n_total=n_total, n_evidence=n_total, 
                     save_every_n = save_every_n, resume = resume)
    # samples = np.load(f'{save_dir}/pocomc_chain_{n_total}.npy')

    return samples, md
# ====================================================================================

def mcmc_wMD_kernel1(exp_active_dict, model_predict, save_dir, n_total, resume):
    """
    With model discrepancy kernel I
    """
    # Define kernel dict compatible with ModelDiscrepancy for active observables 
    kernel_1 = lambda X1, X2, cbar, l, r, s: MD_kernel(X1, X2, cbar, l, r, s)
    HP_log_priors = [log_prior_cbar, log_prior_l, log_prior_r, log_prior_s]
    
    MDkernels = {obs: {"kernel": kernel_1, "log_priors": HP_log_priors}
                 for obs, flag in observables.items() if flag}
        
    # Instantiate class
    md = MD(exp_data=exp_active_dict, 
            model_predict=model_predict, 
            log_priors_model=log_prior_theta,
            MD=True, MDkernels=MDkernels)
    
    # Model parameters bounds (for theta):
    min_theta = model_param_bounds[:, 0]
    max_theta = model_param_bounds[:, 1]
    # Hyperparameters bounds (for phi):
    min_phi = []
    max_phi = []
    for obs in MDkernels:
        n_hp = len(MDkernels[obs]["log_priors"])  # number of hyperparameters for this observable
        # Use the first n_hp rows of HP_bounds for this observable
        min_phi.append(HP_bounds[:n_hp, 0])
        max_phi.append(HP_bounds[:n_hp, 1])
    min_phi = np.concatenate(min_phi)
    max_phi = np.concatenate(max_phi)
    # bounds for all parameters
    min_param = np.concatenate([min_theta, min_phi])
    max_param = np.concatenate([max_theta, max_phi])

    num_param = len(min_param)
    n_effective = n_effective_mltpl * num_param
    n_active = n_active_mltpl * num_param
    n_steps = n_steps_mltpl * num_param

    samples = pocomc(min_param, max_param, 
                     log_posterior=md.log_posterior, samples_save_dir = f"{save_dir}", 
                     n_effective=n_effective, n_active=n_effective, 
                     n_steps=n_steps, n_total=n_total, n_evidence=n_total, 
                     save_every_n = save_every_n, resume = resume)
    # samples = np.load(f'{save_dir}/pocomc_chain_{n_total}.npy')

    return samples, md
# ====================================================================================

def mcmc_wMD_kernel2(exp_active_dict, model_predict, save_dir, n_total, resume):
    """
    With model discrepancy kernel II
    """
    # Define kernel dict compatible with ModelDiscrepancy for active observables 
    kernel_2 = lambda X1, X2, cbar, l, r: MD_kernel(X1, X2, cbar, l, r, s=0)
    HP_log_priors = [log_prior_cbar, log_prior_l, log_prior_r]
    
    MDkernels = {obs: {"kernel": kernel_2, "log_priors": HP_log_priors}
                 for obs, flag in observables.items() if flag}
    
    # Instantiate class
    md = MD(exp_data=exp_active_dict, 
            model_predict=model_predict, 
            log_priors_model=log_prior_theta,
            MD=True, MDkernels=MDkernels)
    
    # Model parameters bounds (for theta):
    min_theta = model_param_bounds[:, 0]
    max_theta = model_param_bounds[:, 1]
    # Hyperparameters bounds (for phi):
    min_phi = []
    max_phi = []
    for obs in MDkernels:
        n_hp = len(MDkernels[obs]["log_priors"])  # number of hyperparameters for this observable
        # Use the first n_hp rows of HP_bounds for this observable
        min_phi.append(HP_bounds[:n_hp, 0])
        max_phi.append(HP_bounds[:n_hp, 1])
    min_phi = np.concatenate(min_phi)
    max_phi = np.concatenate(max_phi)
    # bounds for all parameters
    min_param = np.concatenate([min_theta, min_phi])
    max_param = np.concatenate([max_theta, max_phi])
    
    num_param = len(min_param)
    n_effective = n_effective_mltpl * num_param
    n_active = n_active_mltpl * num_param
    n_steps = n_steps_mltpl * num_param
    
    samples = pocomc(min_param, max_param, 
                     log_posterior=md.log_posterior, samples_save_dir = f"{save_dir}", 
                     n_effective=n_effective, n_active=n_effective, 
                     n_steps=n_steps, n_total=n_total, n_evidence=n_total, 
                     save_every_n = save_every_n, resume = resume)
    # samples = np.load(f'{save_dir}/pocomc_chain_{n_total}.npy')
    
    return samples, md
# ====================================================================================


## Sampling
Load experimental data : `experimental_data.dat`
1) Define active observables in bayesian inference : `observables`
2) Load experimental data and create active experimental data dictionary : `exp_active_dict`
3) Define physics model class and function for model prediction for the active observables to be passed for sampling: `model_predict`
4) Define functions for observable posterior predictive distributions for plotting using samples.
5) Pass `observables`, `exp_active_dict`, `model_predict` to `mcmc_*` function for sampling.
6) Get quantiles for model predictions and model + GP predictions.

### Save mcmc chain considering Bayesian inference with only height

In [ ]:
# Load experimental data 
experimental_data = np.loadtxt('experimental_data.dat')

x_values = experimental_data[:, 0]

In [ ]:
# 1) Define active observables in bayesian inference : `observables`

RunDataDir = "mcmc_data/height"  # Directory to save results
setup_logging(f"logs/height.log")

# Set True to consider in analysis, else set False.
observables = {
    'height': True,
    'velocity': False,
    'acceleration': False
}
# ====================================================================================

# 2) Create active experimental data dictionary : `exp_active_dict`

exp_dict = {}  # Create the dictionary structure

# Iterate over observables
for i, obs in enumerate(observables):
    mean_values = experimental_data[:, 2*i + 1]  # Mean values
    std_values = experimental_data[:, 2*i + 2]   # Std values
    
    exp_dict[obs] = {
        "x": x_values.tolist(),
        "mean": mean_values.tolist(),
        "std": std_values.tolist()
    }
# Dictionary to pass to ModelDiscrepancy class. Remove observables that are False
exp_active_dict = {obs: data for obs, data in exp_dict.items() if observables[obs]}
# ====================================================================================

# 3) Define function for model prediction for the active observables

Model = BallDrop(m=m_true, D=D_true, beta=0, gamma=0)
phy_mod = phy_model(Model=Model, observables=observables, t_obs=t_obs, h0=60.0, t0=0.0)
model_predict = lambda theta: phy_mod.predict(theta)
# model_predict([10,0])
# ====================================================================================

# 4) For getting predictive distributions for observables
phy_mod_plot = phy_model(Model=Model, observables=observables, t_obs=t_plot, h0=60.0, t0=0.0)
model_predict_plot = lambda theta: phy_mod_plot.predict_all(theta)
model_predict_plotMD = lambda theta: phy_mod_plot.predict(theta)

# 5, 6) MCMC sampling and observable quantiles

# wo_MD
save_dir = f"{RunDataDir}/wo_MD"  # Directory to save results
samples, md = mcmc_woMD(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")

# w_MD_kernel1
save_dir = f"{RunDataDir}/w_MD_kernel1"  # Directory to save results
samples, md = mcmc_wMD_kernel1(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")
quantiles_modelPlusGP(samples=samples, MDclass = md, x_predict=t_plot, model_predict_func=model_predict_plotMD, 
                      save_filename=f"{save_dir}/quantiles_20000_modelPlusGP.txt")

# w_MD_kernel2
save_dir = f"{RunDataDir}/w_MD_kernel2"  # Directory to save results
samples, md = mcmc_wMD_kernel2(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")
quantiles_modelPlusGP(samples=samples, MDclass = md, x_predict=t_plot, model_predict_func=model_predict_plotMD, 
                      save_filename=f"{save_dir}/quantiles_20000_modelPlusGP.txt")


### Save mcmc chain considering Bayesian inference with height and velocity

In [ ]:
# 1) Define active observables in bayesian inference : `observables`

RunDataDir = "mcmc_data/height_vel"  # Directory to save results
setup_logging(f"logs/height_vel.log")

# Set True to consider in analysis, else set False.
observables = {
    'height': True,
    'velocity': True,
    'acceleration': False
}
# ====================================================================================

# 2) Create active experimental data dictionary : `exp_active_dict`

exp_dict = {}  # Create the dictionary structure

# Iterate over observables
for i, obs in enumerate(observables):
    mean_values = experimental_data[:, 2*i + 1]  # Mean values
    std_values = experimental_data[:, 2*i + 2]   # Std values
    
    exp_dict[obs] = {
        "x": x_values.tolist(),
        "mean": mean_values.tolist(),
        "std": std_values.tolist()
    }
# Dictionary to pass to ModelDiscrepancy class. Remove observables that are False
exp_active_dict = {obs: data for obs, data in exp_dict.items() if observables[obs]}
# ====================================================================================

# 3) Define function for model prediction for the active observables

Model = BallDrop(m=m_true, D=D_true, beta=0, gamma=0)
phy_mod = phy_model(Model=Model, observables=observables, t_obs=t_obs, h0=60.0, t0=0.0)
model_predict = lambda theta: phy_mod.predict(theta)
# model_predict([10,0])
# ====================================================================================

# 4) For getting predictive distributions for observables

phy_mod_plot = phy_model(Model=Model, observables=observables, t_obs=t_plot, h0=60.0, t0=0.0)
model_predict_plot = lambda theta: phy_mod_plot.predict_all(theta)
model_predict_plotMD = lambda theta: phy_mod_plot.predict(theta)

# 5, 6) MCMC sampling and observable quantiles

# wo_MD
save_dir = f"{RunDataDir}/wo_MD"  # Directory to save results
samples, md = mcmc_woMD(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")

# w_MD_kernel1
save_dir = f"{RunDataDir}/w_MD_kernel1"  # Directory to save results
samples, md = mcmc_wMD_kernel1(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")
quantiles_modelPlusGP(samples=samples, MDclass = md, x_predict=t_plot, model_predict_func=model_predict_plotMD, 
                      save_filename=f"{save_dir}/quantiles_20000_modelPlusGP.txt")

# w_MD_kernel2
save_dir = f"{RunDataDir}/w_MD_kernel2"  # Directory to save results
samples, md = mcmc_wMD_kernel2(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")
quantiles_modelPlusGP(samples=samples, MDclass = md, x_predict=t_plot, model_predict_func=model_predict_plotMD, 
                      save_filename=f"{save_dir}/quantiles_20000_modelPlusGP.txt")


### Save mcmc chain considering Bayesian inference with height, velocity, and acceleration

In [ ]:
# 1) Define active observables in bayesian inference : `observables`

RunDataDir = "mcmc_data/height_vel_acc"  # Directory to save results
setup_logging(f"logs/height_vel_acc.log")

# Set True to consider in analysis, else set False.
observables = {
    'height': True,
    'velocity': True,
    'acceleration': True
}
# ====================================================================================

# 2) Create active experimental data dictionary : `exp_active_dict`

exp_dict = {}  # Create the dictionary structure

# Iterate over observables
for i, obs in enumerate(observables):
    mean_values = experimental_data[:, 2*i + 1]  # Mean values
    std_values = experimental_data[:, 2*i + 2]   # Std values
    
    exp_dict[obs] = {
        "x": x_values.tolist(),
        "mean": mean_values.tolist(),
        "std": std_values.tolist()
    }
# Dictionary to pass to ModelDiscrepancy class. Remove observables that are False
exp_active_dict = {obs: data for obs, data in exp_dict.items() if observables[obs]}
# ====================================================================================

# 3) Define function for model prediction for the active observables

Model = BallDrop(m=m_true, D=D_true, beta=0, gamma=0)
phy_mod = phy_model(Model=Model, observables=observables, t_obs=t_obs, h0=60.0, t0=0.0)
model_predict = lambda theta: phy_mod.predict(theta)
# model_predict([10,0])
# ====================================================================================

# 4) For getting predictive distributions for observables

phy_mod_plot = phy_model(Model=Model, observables=observables, t_obs=t_plot, h0=60.0, t0=0.0)
model_predict_plot = lambda theta: phy_mod_plot.predict_all(theta)
model_predict_plotMD = lambda theta: phy_mod_plot.predict(theta)

# 5, 6) MCMC sampling and observable quantiles

# wo_MD
save_dir = f"{RunDataDir}/wo_MD"  # Directory to save results
samples, md = mcmc_woMD(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")

# w_MD_kernel1
save_dir = f"{RunDataDir}/w_MD_kernel1"  # Directory to save results
samples, md = mcmc_wMD_kernel1(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")
quantiles_modelPlusGP(samples=samples, MDclass = md, x_predict=t_plot, model_predict_func=model_predict_plotMD, 
                      save_filename=f"{save_dir}/quantiles_20000_modelPlusGP.txt")

# w_MD_kernel2
save_dir = f"{RunDataDir}/w_MD_kernel2"  # Directory to save results
samples, md = mcmc_wMD_kernel2(exp_active_dict = exp_active_dict, model_predict = model_predict, save_dir = save_dir, n_total = 20000, resume = False)
quantiles_model(samples=samples, x_predict=t_plot, model_predict_func=model_predict_plot, save_filename=f"{save_dir}/quantiles_20000_model.txt")
quantiles_modelPlusGP(samples=samples, MDclass = md, x_predict=t_plot, model_predict_func=model_predict_plotMD, 
                      save_filename=f"{save_dir}/quantiles_20000_modelPlusGP.txt")
